
# 02 — Prepare the combined proteomics analysis matrix

This notebook maps all VAE and univariate protein targets to AoU Olink assays, loads only target-relevant rows, and creates one PC-adjusted analysis matrix.

Mapping priority:

1. Exact UniProt accession.
2. Exact/known-alias Olink assay symbol from the gene name.
3. If several assays match, choose the one with the largest number of QC-passing participants and retain alternatives in the audit table.

The outcome columns are z-scored NPX values. The same shared disease flags and ancestry PCs are used later for chemistry.


In [ ]:
from pathlib import Path
import os
import sys

# Jupyter's file-browser directory and the kernel working directory can differ.
# Locate the project by its marker instead of trusting Path.cwd().
def find_project_root():
    marker_name = ".aou_combined_replication"
    project_name = "Combined_VAE_linear_AoU_replication"

    env_root = os.getenv("AOU_COMBINED_REPLICATION_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / marker_name).exists():
            return root
        raise FileNotFoundError(
            f"AOU_COMBINED_REPLICATION_ROOT={root} lacks {marker_name}"
        )

    here = Path.cwd().resolve()
    ancestors = [here, *here.parents]
    for candidate in ancestors:
        if (candidate / marker_name).exists():
            return candidate

    matches = []
    for parent in ancestors[:5]:
        patterns = [
            f"*/{marker_name}",
            f"{project_name}/{marker_name}",
            f"*/{project_name}/{marker_name}",
        ]
        for pattern in patterns:
            matches.extend(path.parent for path in parent.glob(pattern))

    matches = sorted(set(path.resolve() for path in matches))
    named = [path for path in matches if path.name == project_name]
    matches = named or matches
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(
        "Could not locate the combined replication project. Set "
        "AOU_COMBINED_REPLICATION_ROOT explicitly. Candidates found: "
        + str(matches)
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import project_config as cfg
from src import aou_replication_utils as aru

print("Kernel working directory:", Path.cwd().resolve())
print("Project root:", cfg.PROJECT_ROOT)
print("Workspace root:", cfg.WORKSPACE_ROOT)


In [ ]:

import numpy as np
import pandas as pd
from IPython.display import display
from google.cloud import bigquery

targets_file = cfg.TARGET_DIR / "analysis_targets_grouped.tsv"
shared_flags_file = cfg.SHARED_DIR / "aou_disease_flag_hits.tsv.gz"
pcs_file = cfg.SHARED_DIR / "aou_genetic_ancestry_pcs.tsv"

for required in [targets_file, shared_flags_file, pcs_file]:
    if not required.exists():
        raise FileNotFoundError(
            f"Missing {required}. Run notebooks 00 and 01 first."
        )

targets = pd.read_csv(targets_file, sep="	")
protein_targets = targets[
    targets["omic_type"].eq("proteomics")
    & targets["clinically_testable"].fillna(False)
].copy()

proteomics_file = cfg.first_existing(cfg.PROTEOMICS_FILE_CANDIDATES)
manifest_file = cfg.first_existing(
    cfg.PROTEOMICS_MANIFEST_CANDIDATES,
    required=False,
)

print(f"Testable protein targets: {len(protein_targets):,}")
print("Proteomics file:", proteomics_file)
print("Manifest file:", manifest_file)

In [ ]:

# Scan the large Olink file once and retain only target-relevant QC-passing rows.
olink_cache = cfg.PROTEOMICS_DIR / "target_olink_rows_qc_pass.tsv.gz"
value_col_note = cfg.PROTEOMICS_DIR / "olink_value_column_used.txt"

if olink_cache.exists() and not cfg.FORCE_REBUILD:
    olink_rows = pd.read_csv(olink_cache, sep="	", compression="gzip")
    value_col = value_col_note.read_text().strip() if value_col_note.exists() else "unknown"
    print("Reused target Olink cache:", olink_cache)
else:
    olink_rows, value_col = aru.scan_target_proteomics_rows(
        proteomics_file,
        protein_targets,
        chunksize=1_000_000,
    )
    olink_rows.to_csv(
        olink_cache,
        sep="	",
        index=False,
        compression="gzip",
    )
    value_col_note.write_text(value_col, encoding="utf-8")
    print("Saved target Olink cache:", olink_cache)

print(f"Retained QC-passing target Olink rows: {len(olink_rows):,}")
print("Olink value column:", value_col)


In [ ]:

protein_mapping = aru.select_protein_assays(protein_targets, olink_rows)
protein_mapping.to_csv(
    cfg.PROTEOMICS_DIR / "protein_target_to_aou_assay_mapping.tsv",
    sep="	",
    index=False,
)

mapped_n = protein_mapping["aou_assay"].notna().sum()
print(f"Unique proteins mapped: {mapped_n:,} / {len(protein_mapping):,}")
display(
    protein_mapping.sort_values(
        ["protein_mapping_method", "aou_n_people_qc", "feature_display"],
        ascending=[True, False, True],
    )
)

protein_z = aru.build_protein_z_matrix(olink_rows, protein_mapping)
if protein_z.empty:
    raise RuntimeError("No protein outcome columns were created")

print("Protein z-score matrix:", protein_z.shape)


In [ ]:
# -----------------------------------------------------------------------------
# Query person covariates for participants present in the protein matrix
#
# This deliberately does NOT use cfg.PREP or a workspace-owned BigQuery table.
# The participant IDs are already available locally in protein_z.
# -----------------------------------------------------------------------------

import warnings

client = bigquery.Client(project=cfg.GOOGLE_PROJECT)

demographics_out = (
    cfg.PROTEOMICS_DIR
    / "proteomics_person_covariates.tsv"
)

queried_ids_out = (
    cfg.PROTEOMICS_DIR
    / "proteomics_person_covariate_query_ids.tsv"
)

# Extract the exact participant set represented in the protein outcome matrix.
person_ids = (
    pd.to_numeric(
        protein_z["person_id"],
        errors="coerce",
    )
    .dropna()
    .astype("int64")
    .drop_duplicates()
    .sort_values()
    .tolist()
)

if not person_ids:
    raise RuntimeError(
        "No valid person IDs were found in protein_z."
    )

expected_id_set = set(person_ids)

print(
    "Proteomics participants requiring covariates:",
    f"{len(person_ids):,}",
)

# Parameterized query. @person_ids is supplied separately through
# bigquery.ArrayQueryParameter rather than interpolated into the SQL text.
demographics_query = f"""
SELECT
  p.person_id,
  p.year_of_birth,
  p.birth_datetime,
  p.sex_at_birth_concept_id,
  sex_birth.concept_name AS sex_at_birth,
  p.gender_concept_id,
  gender.concept_name AS gender,
  p.race_concept_id,
  race.concept_name AS race,
  p.ethnicity_concept_id,
  ethnicity.concept_name AS ethnicity
FROM `{cfg.CDR}.person` AS p
LEFT JOIN `{cfg.CDR}.concept` AS sex_birth
  ON p.sex_at_birth_concept_id = sex_birth.concept_id
LEFT JOIN `{cfg.CDR}.concept` AS gender
  ON p.gender_concept_id = gender.concept_id
LEFT JOIN `{cfg.CDR}.concept` AS race
  ON p.race_concept_id = race.concept_id
LEFT JOIN `{cfg.CDR}.concept` AS ethnicity
  ON p.ethnicity_concept_id = ethnicity.concept_id
WHERE p.person_id IN UNNEST(@person_ids)
ORDER BY p.person_id
"""

# Save the reusable SQL template for troubleshooting.
(cfg.LOG_DIR / "proteomics_person_covariates.sql").write_text(
    demographics_query,
    encoding="utf-8",
)

# -------------------------------------------------------------------------
# Decide whether the cached covariate table matches the current participant set
# -------------------------------------------------------------------------

use_cache = (
    demographics_out.exists()
    and queried_ids_out.exists()
    and not cfg.FORCE_REBUILD
)

if use_cache:
    previous_ids = pd.read_csv(
        queried_ids_out,
        sep="\t",
    )

    if "person_id" not in previous_ids.columns:
        use_cache = False
    else:
        previous_id_set = set(
            pd.to_numeric(
                previous_ids["person_id"],
                errors="coerce",
            )
            .dropna()
            .astype("int64")
        )

        if previous_id_set != expected_id_set:
            print(
                "Cached covariate query used a different participant set; "
                "rebuilding."
            )
            use_cache = False

# -------------------------------------------------------------------------
# Read cache or query BigQuery
# -------------------------------------------------------------------------

if use_cache:
    people = pd.read_csv(
        demographics_out,
        sep="\t",
    )

    print(
        "Reused cached proteomics covariates:",
        demographics_out,
    )

else:
    # Chunking keeps the array parameter manageable and makes failures easier
    # to identify. The Olink cohort should require only a few chunks.
    QUERY_CHUNK_SIZE = 5_000
    frames = []

    for start in range(
        0,
        len(person_ids),
        QUERY_CHUNK_SIZE,
    ):
        stop = min(
            start + QUERY_CHUNK_SIZE,
            len(person_ids),
        )

        id_chunk = [
            int(value)
            for value in person_ids[start:stop]
        ]

        job_config = bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ArrayQueryParameter(
                    "person_ids",
                    "INT64",
                    id_chunk,
                )
            ]
        )

        print(
            f"Querying covariates for IDs "
            f"{start + 1:,}-{stop:,} of {len(person_ids):,}..."
        )

        chunk_result = client.query(
            demographics_query,
            job_config=job_config,
        ).to_dataframe()

        frames.append(chunk_result)

    if not frames:
        raise RuntimeError(
            "No BigQuery covariate-query chunks were executed."
        )

    people = pd.concat(
        frames,
        ignore_index=True,
    )

    if people.empty:
        raise RuntimeError(
            "The covariate query returned zero participants."
        )

    # Save both results and the exact requested participant set.
    people.to_csv(
        demographics_out,
        sep="\t",
        index=False,
    )

    pd.DataFrame(
        {"person_id": person_ids}
    ).to_csv(
        queried_ids_out,
        sep="\t",
        index=False,
    )

    print(
        "Saved proteomics covariates:",
        demographics_out,
    )

# -------------------------------------------------------------------------
# Validate returned covariates
# -------------------------------------------------------------------------

people["person_id"] = pd.to_numeric(
    people["person_id"],
    errors="raise",
).astype("int64")

if people["person_id"].duplicated().any():
    duplicated_ids = (
        people.loc[
            people["person_id"].duplicated(
                keep=False
            ),
            "person_id",
        ]
        .drop_duplicates()
        .tolist()
    )

    raise ValueError(
        "Proteomics covariate query returned duplicate people. "
        f"Example duplicated IDs: {duplicated_ids[:10]}"
    )

returned_id_set = set(
    people["person_id"]
)

unexpected_ids = (
    returned_id_set
    - expected_id_set
)

if unexpected_ids:
    raise ValueError(
        "The covariate query returned IDs that were not requested. "
        f"Examples: {sorted(unexpected_ids)[:10]}"
    )

missing_ids = (
    expected_id_set
    - returned_id_set
)

if missing_ids:
    warnings.warn(
        f"{len(missing_ids):,} proteomics participant IDs were not found "
        "in the CDR person table. They will be excluded when the analysis "
        "matrix is assembled."
    )

print(
    "Covariate rows returned:",
    f"{len(people):,}",
)

print(
    "Requested IDs missing from person table:",
    f"{len(missing_ids):,}",
)

# -------------------------------------------------------------------------
# Construct sex covariate
# -------------------------------------------------------------------------

# Prefer sex at birth for adjustment; use gender only as a fallback.
sex_raw = (
    people["sex_at_birth"]
    .fillna(people["gender"])
    .fillna("Other/Unknown")
)

sex_lower = (
    sex_raw
    .astype(str)
    .str.strip()
    .str.lower()
)

people["sex_adjustment"] = np.select(
    [
        sex_lower.eq("female"),
        sex_lower.eq("male"),
    ],
    [
        "Female",
        "Male",
    ],
    default="Other/Unknown",
)

display(
    people.head()
)

In [ ]:

# Assign the best available date for age calculation.
if manifest_file is not None:
    assay_dates = aru.parse_manifest_assay_dates(manifest_file)
    people = people.merge(assay_dates, on="person_id", how="left", validate="one_to_one")
else:
    people["assay_date"] = pd.NaT

observed_dates = pd.to_datetime(people["assay_date"], errors="coerce")
fallback_date = observed_dates.median()
if pd.isna(fallback_date):
    fallback_date = pd.Timestamp("2024-06-15")
people["assay_date_source"] = np.where(
    observed_dates.notna(), "manifest_or_collection_date", "fallback_median_date"
)
people["assay_date"] = observed_dates.fillna(fallback_date)
people = aru.calculate_age_at_date(
    people,
    date_col="assay_date",
    output_col="age_at_assay",
)

# Exclude impossible ages rather than allowing them to distort spline fits.
people.loc[~people["age_at_assay"].between(18, 110), "age_at_assay"] = np.nan

flag_hits = pd.read_csv(shared_flags_file, sep="	", compression="gzip")
flag_hits["person_id"] = pd.to_numeric(flag_hits["person_id"], errors="raise").astype("int64")
disease_defs = aru.load_disease_definitions(cfg.CONFIG_DIR / "disease_definitions.json")
base_domains = [item["domain"] for item in disease_defs]
flags = aru.add_case_control_flags(people, flag_hits, base_domains)

pcs = pd.read_csv(pcs_file, sep="	")
pcs["person_id"] = pd.to_numeric(pcs["person_id"], errors="raise").astype("int64")

matrix = (
    people.merge(flags, on="person_id", how="left", validate="one_to_one")
    .merge(pcs, on="person_id", how="left", validate="one_to_one")
    .merge(protein_z, on="person_id", how="inner", validate="one_to_one")
)
matrix["has_complete_pc8"] = matrix[cfg.PC_COLUMNS].notna().all(axis=1)

matrix_out = cfg.PROTEOMICS_DIR / "proteomics_analysis_matrix.tsv.gz"
matrix.to_csv(matrix_out, sep="	", index=False, compression="gzip")
print("Saved matrix:", matrix_out)
print("Matrix shape:", matrix.shape)
print("Complete PC8:", int(matrix["has_complete_pc8"].sum()))


In [ ]:

# Create the feature-key -> matrix-column map used by the common model notebook.
feature_map = protein_mapping[
    [
        "feature_key",
        "aou_assay",
        "protein_mapping_method",
        "aou_n_people_qc",
        "n_mapping_candidates",
        "alternative_assays",
    ]
].rename(columns={"aou_assay": "outcome_col"})
feature_map["available_for_model"] = feature_map["outcome_col"].notna()
feature_map.to_csv(
    cfg.PROTEOMICS_DIR / "proteomics_feature_column_map.tsv",
    sep="	",
    index=False,
)

# Cohort-specific case/control counts, including the PC-complete primary subset.
summary_rows = []
pc_mask = matrix["has_complete_pc8"]
for domain in aru.CASE_CONTROL_DOMAINS:
    summary_rows.append(
        {
            "analysis_domain": domain,
            "n_total": len(matrix),
            "n_cases": int(matrix[f"{domain}_case"].sum()),
            "n_domain_controls": int(
                matrix[f"{domain}_domain_control_candidate"].sum()
            ),
            "n_cases_complete_pc8": int(
                matrix.loc[pc_mask, f"{domain}_case"].sum()
            ),
            "n_domain_controls_complete_pc8": int(
                matrix.loc[pc_mask, f"{domain}_domain_control_candidate"].sum()
            ),
        }
    )
cohort_summary = pd.DataFrame(summary_rows)
cohort_summary.to_csv(
    cfg.PROTEOMICS_DIR / "proteomics_case_control_summary.tsv",
    sep="	",
    index=False,
)
display(cohort_summary)
